Use this notebook to prepare input data for de novo cases (json file and structures) --NanoDesigner

Skip until part two the notebook if the epitope informations is to be extracted from another nanobody-antigen or antibody-antigen PDB complex.


In [ ]:
# use nanodesigner1 kernel
import sys
import os
import shutil
import json
import traceback
sys.path.append("./NanoDesigner")  # Add the NanoDesigner directory to the Python path
from functionalities.complex_analysis import (
    fetch_from_sabdab, fetch_from_pdb, extract_antibody_info,
    get_cdr_pos, extract_seq_info_from_pdb
)
from dyMEAN.data.pdb_utils import Protein  
from dyMEAN.utils.renumber import renumber_pdb
from functionalities.nanobody_antibody_interacting_residues import interacting_residues, interacting_residues_extended


In [ ]:
# Fill in this dictionary with the required information:
config = {
    "Nano_source_pdb": "6lr7",
    "nano_chain_id": "B", #heavy chain
    "light_chain_id": "", # for nanobodies, define empty string ""
    "Antigen_source_pdb": "4obe",
    "antigen_chain_id":["A"],
    "epitope_sequences": ["NHFVDEYDPTIEDSYR","TAGQEEYSAMRDQYMRTGE", "YKLV;CLL"],  # defined by the user
    "out_dir": "./NanoDesigner/your_working_directory/denovo_epitope_info",
    "numbering_scheme" : "imgt", # imgt or chothia
  }


In [ ]:
# Set the output directory and numbering scheme
out_dir = config["out_dir"]
os.makedirs(out_dir, exist_ok=True)

# Create a temporary directory within the output directory
tmp_dir = os.path.join(out_dir, "temporal_files")
os.makedirs(tmp_dir, exist_ok=True)

# Define the directory to look for pre-existing structures
all_structures_dir = "./NanoDesigner/all_structures" # UPDATE
numbering_scheme = config["numbering_scheme"]

print("Output directory created:", out_dir)
print("Temporary directory created:", tmp_dir)

Output directory created: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info
Temporary directory created: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/temporal_files


In [ ]:
# Define nanobody and antigen PDB paths
nano = config["Nano_source_pdb"]
nano_id = config["nano_chain_id"]
numbering_scheme = config["numbering_scheme"]

# Temporary paths where files will be copied or downloaded
nano_source = os.path.join(tmp_dir, nano + "_raw.pdb")
antigen = config["Antigen_source_pdb"]
antigen_id = config["antigen_chain_id"]
antigen_source = os.path.join(tmp_dir, antigen + "_raw.pdb")

# Check if the nanobody PDB is already available in all_structures
all_structures_dir = "./NanoDesigner/all_structures"
nano_local_path = os.path.join(all_structures_dir, numbering_scheme, nano + ".pdb")

if os.path.exists(nano_local_path):
    # Copy nanobody PDB from all_structures if it exists
    print(f"Found nanobody PDB locally at {nano_local_path}. Copying to temporary directory.")
    shutil.copy(nano_local_path, nano_source)
else:
    # Download the nanobody PDB if it's not available locally
    print("Nanobody PDB not found locally. Attempting to download from SAbDab.")
    fetch_from_sabdab(nano, numbering_scheme, nano_source, tries=5)
    if not os.path.exists(nano_source):
        raise FileNotFoundError(f"Could not fetch PDB for {nano} from SAbDab.")

# Download the antigen PDB from PDB directly
print("Attempting to download antigen PDB from PDB database.")
fetch_from_pdb(antigen, antigen_source, tries=5)
if not os.path.exists(antigen_source):
    raise FileNotFoundError(f"Could not fetch PDB for {antigen} from PDB website.")


Found nanobody PDB locally at /home/rioszemm/NanoDesigner/all_structures/imgt/6lr7.pdb. Copying to temporary directory.
Attempting to download antigen PDB from PDB database.


In [ ]:
# Define nanobody PDB paths
nano = config["Nano_source_pdb"]
nano_id = config["nano_chain_id"]
nano_local_path = os.path.join(all_structures_dir, numbering_scheme, nano + ".pdb")
nano_source = os.path.join(tmp_dir, nano + "_raw.pdb")

# Check if nanobody PDB is already available locally
if os.path.exists(nano_local_path):
    print(f"Found nanobody PDB locally at {nano_local_path}. Copying to temporary directory.")
    shutil.copy(nano_local_path, nano_source)
else:
    print("Nanobody PDB not found locally. Attempting to download from SAbDab.")
    fetch_from_sabdab(nano, numbering_scheme, nano_source, tries=5)
    if not os.path.exists(nano_source):
        raise FileNotFoundError(f"Could not fetch PDB for {nano} from SAbDab.")

Found nanobody PDB locally at /home/rioszemm/NanoDesigner/all_structures/imgt/6lr7.pdb. Copying to temporary directory.


In [22]:
# Initialize JSON content with basic information
json_content = {
    "pdb": nano + f"_{antigen}",
    "heavy_chain": nano_id,
    "light_chain": config["light_chain_id"],
    "antigen_chains": antigen_id
}

# Path for cleaned antigen PDB file
antigen_ = os.path.join(tmp_dir, antigen + ".pdb")
json_content["antigen_source"] = antigen_

# Extract antigen sequence
try:
    pdb = Protein.from_pdb(antigen_source, antigen_id)  # Assume antigen_id is a list
    json_content['antigen_seqs'] = []

    for peptide_id, peptide in pdb.peptides.items():
        sequence = peptide.get_seq()
        print(f"Antigen Peptide ID: {peptide_id}, Sequence: {sequence}")
        json_content['antigen_seqs'].append(sequence)

    # Save the cleaned antigen structure
    pdb.to_pdb(antigen_)
except Exception as e:
    print(f"Error processing antigen sequence: {e}")


Antigen Peptide ID: A, Sequence: MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEK


In [13]:
# Define cleaned nanobody PDB path
nano_ = os.path.join(tmp_dir, nano + ".pdb")
json_content["nano_source"] = nano_

# Renumber the nanobody and extract its sequence
try:
    renumber_pdb(nano_source, nano_, numbering_scheme)
    pdb = Protein.from_pdb(nano_, [nano_id])  # nano_id should be a list
    json_content['heavy_chain_seq'] = ""

    for peptide_id, peptide in pdb.peptides.items():
        sequence = peptide.get_seq()
        print(f"Nanobody Peptide ID: {peptide_id}, Sequence: {sequence}")
        json_content['heavy_chain_seq'] += sequence

    # Extract CDR positions and sequences
    cdr_pos_dict = extract_antibody_info(pdb, nano_id, "", numbering_scheme)
    nano_peptide = pdb.peptides.get(nano_id)

    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()
        cdr_pos = get_cdr_pos(cdr_pos_dict, cdr_name)
        json_content[f'cdr{cdr_name}_pos'] = cdr_pos
        start, end = cdr_pos
        cdr_seq = nano_peptide.get_span(start, end + 1).get_seq()
        json_content[f'cdr{cdr_name}_seq'] = cdr_seq
except Exception as e:
    print(f"Error processing nanobody sequence: {e}")


chain B type: H
Nanobody Peptide ID: B, Sequence: VQLVESGGRLVQAGDSLRLSCAASGRTFSTSAMAWFRQAPGREREFVAAITWTVGNTILGDSVKGRFTISRDRAKNTVDLQMDNLEPEDTAVYYCSARSRGYVLSVLRSVDSYDYWGQGTQVTVS


In [ ]:
# Define the output directory for the nanobody-antigen combination
nanobody_id = config["Nano_source_pdb"]
antigen_id = config["Antigen_source_pdb"]
output_dir = os.path.join(out_dir, f"{nanobody_id}_{antigen_id}")
os.makedirs(output_dir, exist_ok=True)
print(f"Saving JSON files to directory: {output_dir}")

# Prepare paths
nano_source = os.path.join(tmp_dir, f"{nanobody_id}.pdb")
antigen_source = os.path.join(tmp_dir, f"{antigen_id}.pdb")
numbering_scheme = config["numbering_scheme"]

# Initialize JSON content for shared fields
json_content = {
    "pdb": f"{nanobody_id}_{antigen_id}",
    "heavy_chain": config["nano_chain_id"],
    "light_chain": config["light_chain_id"],
    "antigen_chains": config["antigen_chain_id"],
    "antigen_source": antigen_source,
    "nano_source": nano_source
}

try:
    ag_chains_to_reconstruct = config["antigen_chain_id"]
    pdb = Protein.from_pdb(antigen_source, ag_chains_to_reconstruct)
    json_content['antigen_seqs'] = [peptide.get_seq() for peptide_id, peptide in pdb.peptides.items()]
    
    # Save cleaned antigen structure
    pdb.to_pdb(antigen_source)
except Exception as e:
    print(f"Error processing antigen sequence: {e}")
    print(traceback.format_exc())

try:
    renumber_pdb(nano_source, nano_source, numbering_scheme)    
    pdb = Protein.from_pdb(nano_source, [config["nano_chain_id"]])
    json_content['heavy_chain_seq'] = "".join(peptide.get_seq() for peptide_id, peptide in pdb.peptides.items())
    
    # Extract CDR positions
    cdr_pos_dict = extract_antibody_info(pdb, config["nano_chain_id"], config["light_chain_id"], numbering_scheme)
    
    # Get CDR sequences using CDR positions
    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()  # e.g., "h1", "h2", "h3"
        cdr_key = f"cdr{cdr_name}_pos"
        cdr_seq_key = f"cdr{cdr_name}_seq"
        
        cdr_pos = get_cdr_pos(cdr_pos_dict, f'H{i}')
        if cdr_pos:
            json_content[cdr_key] = cdr_pos
            start, end = cdr_pos
            nano_peptide = pdb.peptides.get(config["nano_chain_id"])
            json_content[cdr_seq_key] = nano_peptide.get_span(start, end + 1).get_seq()
except Exception as e:
    print(f"Error processing nanobody sequence and CDR extraction: {e}")
    print(traceback.format_exc())

list_of_dicts = []
for count, epitope in enumerate(config["epitope_sequences"], start=1):
    function_results = []

    # Split epitope sequences if there are multiple segments
    if ";" in epitope:
        epitope_list = epitope.split(";")
        for i, substring in enumerate(epitope_list, start=1):
            epitope_pdb = extract_seq_info_from_pdb(antigen_source, config["antigen_chain_id"][0], substring)
            if epitope_pdb is not None:
                function_results.extend(epitope_pdb)
            else:
                print(f"Warning: Could not extract information for epitope segment '{substring}' in epitope '{epitope}'")
    else:
        epitope_pdb = extract_seq_info_from_pdb(antigen_source, config["antigen_chain_id"][0], epitope)
        if epitope_pdb is not None:
            function_results.extend(epitope_pdb)
        else:
            print(f"Warning: Could not extract information for epitope '{epitope}'")

    # Create a new dictionary with the epitope information
    new_dict = json_content.copy()
    new_dict["numbering"] = numbering_scheme
    new_dict["pdb"] = f"{nanobody_id}_{antigen_id}_ep_{count}"
    new_dict["epitope_user_input"] = "yes"
    new_dict['epitope'] = function_results  # Add the extracted epitope data
    list_of_dicts.append(new_dict)

    # Save each epitope dictionary as a separate JSON file in the output directory
    epitope_filename = os.path.join(output_dir, f"{new_dict['pdb']}.json")
    with open(epitope_filename, 'w') as json_file:
        json.dump(new_dict, json_file)
    print(f"Saved epitope JSON file: {epitope_filename}")

print("All epitope JSON files have been saved.")



Saving JSON files to directory: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/6lr7_4obe
chain B type: H
Saved epitope JSON file: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/6lr7_4obe/6lr7_4obe_ep_1.json
Saved epitope JSON file: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/6lr7_4obe/6lr7_4obe_ep_2.json
Saved epitope JSON file: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/6lr7_4obe/6lr7_4obe_ep_3.json
All epitope JSON files have been saved.


PART 2
Epitope input: DEFINED USING dSASA -from a known antibody-nanobody complex


Example: HER2-Trastuzumab-Pertuzumab complex (PDB_ID:8PWH) from the Protein DataBase
    Trastuzumab Fab light chain: A
    Trastuzumab Fab heavy chain: B
    Pertuzumab Fab light chain: C
    Pertuzumab Fab heavy chain: D

In this case, run the notebook twice, one per antibody/nanobody molecule bound to the antigen.
Using a nanobody targeting Caplacizumab nanobody as scaffold (PDB_ID:7eow) from the Protein DataBase

In [ ]:
# use nanodesigner1 kernel
import sys
import os
import shutil
import json
import traceback
sys.path.append("/NanoDesigner")  # Add the NanoDesigner directory to the Python path
from functionalities.complex_analysis import (
    fetch_from_sabdab, fetch_from_pdb, extract_antibody_info,
    get_cdr_pos
)

from functionalities.nanobody_antibody_interacting_residues import interacting_residues, interacting_residues_extended
from dyMEAN.data.pdb_utils import Protein  
from dyMEAN.utils.renumber import renumber_pdb


In [67]:
# Define the configuration dictionary
config = {
    "Nano_source_pdb": "7eow",  # Nanobody PDB ID to be used as scaffold
    "nano_chain_id": "B",  # Nanobody heavy chain
    "light_chain_id": "",  # Set to empty if nanobody has no light chain
    "Antigen_complex_source_pdb": "8pwh",  # Antigen PDB ID. Antigen bound to either a nanobody or antibody
    "antigen_chain_id": ["E"],  # List of antigen chain IDs involved in interaction, for now use only one-chain antigens
    "immunoglobulin_heavy_chain": "B",  # List of nanobody/antibody chain IDs involved in interaction to the antigen
    "immunoglobulin_light_chain": "A",  # List of nanobody/antibody chain IDs involved in interaction to the antigen
    "out_dir": "/home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info",  # Output directory
    "numbering_scheme": "imgt",  # Numbering scheme to use, e.g., imgt or chothia
    "tmp_dir": "/home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/temporal_files"  # Temporary directory
}


In [68]:
# Set the output and temporary directories
out_dir = config["out_dir"]
os.makedirs(out_dir, exist_ok=True)

tmp_dir = config["tmp_dir"]
os.makedirs(tmp_dir, exist_ok=True)

# Define all_structures directory for local look-up
all_structures_dir = "./NanoDesigner/all_structures"
numbering_scheme = config["numbering_scheme"]

# Define paths for local nanobody PDB and temporary download destination
nano_local_path = os.path.join(all_structures_dir, numbering_scheme, config["Nano_source_pdb"] + ".pdb")
nano_source = os.path.join(tmp_dir, config["Nano_source_pdb"] + "_raw.pdb")
antigen_source = os.path.join(tmp_dir, config["Antigen_complex_source_pdb"] + "_raw.pdb")

print("Output directory created:", out_dir)
print("Temporary directory created:", tmp_dir)


Output directory created: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info
Temporary directory created: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/temporal_files


In [69]:

# Check if the nanobody PDB file exists locally
if os.path.exists(nano_local_path):
    # Copy local file to temporary directory if it exists
    print(f"Found nanobody PDB locally at {nano_local_path}. Copying to temporary directory.")
    shutil.copy(nano_local_path, nano_source)
else:
    # Download the nanobody PDB if not found locally
    print("Nanobody PDB not found locally. Attempting to download from SAbDab.")
    fetch_from_sabdab(config["Nano_source_pdb"], config["numbering_scheme"], nano_source, tries=5)
    if not os.path.exists(nano_source):
        raise FileNotFoundError(f"Could not fetch PDB for {config['Nano_source_pdb']} from SAbDab.")

try:
    print("Attempting to download antigen PDB from PDB database.")
    fetch_from_pdb(config["Antigen_complex_source_pdb"], antigen_source, tries=5)
    print("Antigen PDB download complete.")
except Exception as e:
    print(f"Error downloading antigen PDB: {e}")

Found nanobody PDB locally at /home/rioszemm/NanoDesigner/all_structures/imgt/7eow.pdb. Copying to temporary directory.
Attempting to download antigen PDB from PDB database.


Antigen PDB download complete.


In [70]:
# Define paths for the cleaned-up and renumbered PDB files
nano_renumbered = os.path.join(tmp_dir, config["Nano_source_pdb"] + "_imgt.pdb")
antigen_renumbered = os.path.join(tmp_dir, config["Antigen_complex_source_pdb"] + "_imgt.pdb")

# Renumber nanobody and antigen files using the specified numbering scheme
try:
    print("Renumbering nanobody PDB file...")
    renumber_pdb(nano_source, nano_renumbered, config["numbering_scheme"])
    
    print("Renumbering antigen PDB file...")
    renumber_pdb(antigen_source, antigen_renumbered, config["numbering_scheme"])
    
except Exception as e:
    print(f"Error during renumbering: {e}")

Renumbering nanobody PDB file...
chain B type: H
Renumbering antigen PDB file...
chain A type: K
chain B type: H
chain C type: K
chain D type: H


In [71]:
# Initialize a dictionary to store nanobody information
nanobody_info = {
    "pdb": config["Nano_source_pdb"],
    "heavy_chain": config["nano_chain_id"],
    "light_chain": config["light_chain_id"],
}

numbering = config["numbering_scheme"]

# Load the renumbered nanobody PDB to extract CDR information
try:
    # Load the renumbered nanobody
    pdb_nano = Protein.from_pdb(nano_renumbered, [config["nano_chain_id"]])
    
    # Extract CDR positions using the numbering scheme
    cdr_pos_dict = extract_antibody_info(pdb_nano, config["nano_chain_id"], config["light_chain_id"], config["numbering_scheme"])
    
    # Get the sequences and positions for each CDR
    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()  # e.g., 'h1', 'h2', 'h3'
        cdr_pos = get_cdr_pos(cdr_pos_dict, f'H{i}')
        
        if cdr_pos:
            nanobody_info[f'cdr{cdr_name}_pos'] = cdr_pos
            start, end = cdr_pos
            nano_peptide = pdb_nano.peptides.get(config["nano_chain_id"])
            nanobody_info[f'cdr{cdr_name}_seq'] = nano_peptide.get_span(start, end + 1).get_seq()
    
    print("CDR extraction completed.")

    nanobody_info['heavy_chain_seq'] = ""
    for peptide_id, peptide in pdb_nano.peptides.items():
        sequence = peptide.get_seq()
        print(f"Peptide ID: {peptide_id}, Sequence: {sequence}")
        # Append the sequence to the 'antigen_seqs' list
        nanobody_info['heavy_chain_seq'] += sequence


    original_dir = os.path.dirname(nano_renumbered)
    original_filename = os.path.basename(nano_renumbered)
    cleaned_filename = original_filename.replace(f"_{numbering}", "") 
    cleaned_pdb_path = os.path.join(original_dir, cleaned_filename)
    
    pdb_nano.to_pdb(cleaned_pdb_path)
    nanobody_info["nano_source"] = cleaned_pdb_path

    print("heavy chain sequence extracted")
except Exception as e:
    print(f"Error extracting CDRs: {e}")



# add "heavy_chain_seq" key/value 

CDR extraction completed.
Peptide ID: B, Sequence: EVQLVESGGGLVQPGGSLRLSCAASGRTFSYNPMGWFRQAPGKGRELVAAISRTGGSTYYPDSVEGRFTISRDNAKRMVYLQMNSLRAEDTAVYYCAAAGVRAEDGRVRTLPSEYTFWGQGTQVTVSS
heavy chain sequence extracted


In [72]:
# Initialize a dictionary to store nanobody information
temporal_info = {
    "pdb": config["Antigen_complex_source_pdb"],
    "heavy_chain": config["immunoglobulin_heavy_chain"],
    "light_chain": config["immunoglobulin_light_chain"],
    "antigen_chains": config["antigen_chain_id"]
}

H, L, A = config["immunoglobulin_heavy_chain"], config["immunoglobulin_light_chain"], config["antigen_chain_id"]
print(A)
chains_to_iterate = [H, L] if L != "" else [H]
all_complex_chains = chains_to_iterate + A
numbering = config["numbering_scheme"]

# Load the renumbered nanobody PDB to extract CDR information
try:
    # Load the complex
    pdb_complex = Protein.from_pdb(antigen_renumbered, all_complex_chains)
    
    # Extract CDR positions using the numbering scheme
    cdr_pos_dict = extract_antibody_info(pdb_complex, H, L, numbering)
    
    # Get the sequences and positions for each CDR
    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()  # e.g., 'h1', 'h2', 'h3'
        cdr_pos = get_cdr_pos(cdr_pos_dict, f'H{i}')
        
        if cdr_pos:
            temporal_info[f'cdr{cdr_name}_pos'] = cdr_pos
            start, end = cdr_pos
            nano_peptide = pdb_complex.peptides.get(H)
            temporal_info[f'cdr{cdr_name}_seq'] = nano_peptide.get_span(start, end + 1).get_seq()
    
    if L:
        for i in range(1, 4):
            cdr_name = f'L{i}'.lower()  
            cdr_pos = get_cdr_pos(cdr_pos_dict, f'L{i}')
            
            if cdr_pos:
                temporal_info[f'cdr{cdr_name}_pos'] = cdr_pos
                start, end = cdr_pos
                light_peptide = pdb_complex.peptides.get(L)
                temporal_info[f'cdr{cdr_name}_seq'] = light_peptide.get_span(start, end + 1).get_seq()

    print("CDR extraction completed.")

    ag_chains_to_reconstruct = A    
    for antigen_chain in ag_chains_to_reconstruct:
        try:
            pdb = Protein.from_pdb(antigen_renumbered, ag_chains_to_reconstruct)
            temporal_info['antigen_seqs'] = []
        
            for peptide_id, peptide in pdb.peptides.items():
                sequence = peptide.get_seq()
                print(f"Peptide ID: {peptide_id}, Sequence: {sequence}")
                # Append the sequence to the 'antigen_seqs' list
                temporal_info['antigen_seqs'].append(sequence)

        except Exception as e:
            print(f'Something went wrong for {antigen_renumbered}, {e}')
            print(traceback.format_exc())

    # Modify the path to save the cleaned PDB without "_imgt"
    original_dir = os.path.dirname(antigen_renumbered)
    original_filename = os.path.basename(antigen_renumbered)
    cleaned_filename = original_filename.replace(f"_{numbering}", "") 
    cleaned_pdb_path = os.path.join(original_dir, cleaned_filename)

    # Save the cleaned PDB structure to the new path

    pdb.to_pdb(cleaned_pdb_path)
    print(f"Cleaned PDB file saved to: {cleaned_pdb_path}")
    temporal_info["antigen_source"] = cleaned_pdb_path

except Exception as e:
    print(f"Error during processing: {e}")
    print(traceback.format_exc())

['E']
CDR extraction completed.
Peptide ID: E, Sequence: TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQ
Cleaned PDB file saved to: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/temporal_files/8pwh.pdb


In [73]:
def binding_residues_analysis(temporal_dir, item, complex_pdb_file):
    pdb_file = complex_pdb_file
    heavy_chain = item["heavy_chain"]
    light_chain = item["light_chain"]  # may be an empty string ""
    antigen_chains = item["antigen_chains"]
    pdb_id = item["pdb"]

    # Initialize epitope model to store all interaction results
    epitope_model = []

    # Iterate through each antigen chain to check interactions
    for antigen_chain in antigen_chains:
        for chain in [heavy_chain, light_chain]:
            if not chain:  # Skip if light chain is empty
                continue
            
            chain_type = 'H' if chain == heavy_chain else 'L'
            print(f"Processing antigen {antigen_chain} with {chain_type} chain")

            chains_to_reconstruct = [chain, antigen_chain]
            
            # Generate a unique filename for each antigen-chain pair
            tmp_pdb = os.path.join(temporal_dir, f"{pdb_id}_{chain_type}_chain_{antigen_chain}.pdb")
            if not os.path.exists(tmp_pdb):
                try:
                    # Extract and save PDB file for each chain pair
                    protein = Protein.from_pdb(pdb_file, chains_to_reconstruct)
                    protein.to_pdb(tmp_pdb)
                    assert os.path.exists(tmp_pdb), f"Temporary PDB file not created: {tmp_pdb}"
                except Exception as e:
                    print(f"Failed to process PDB file '{pdb_file}' with {chain_type} chain and antigen {antigen_chain}: {e}")
                    continue

            # Analyze interaction for each antigen-chain pair
            result = interacting_residues_extended(item, tmp_pdb, antigen_chain, temporal_dir, chain_type)
            if result is not None:
                epitope_model.extend(result)  # Collect results if they exist
            
    return epitope_model

In [74]:
tmp_dir_for_analysis = os.path.join(config["tmp_dir"], "dSASA_computations")
os.makedirs(tmp_dir_for_analysis, exist_ok=True)
epitope = binding_residues_analysis(tmp_dir_for_analysis,temporal_info, antigen_renumbered)


Processing antigen E with H chain
Command executed successfully.
Interacting aminoacids computation took: 9.45 seconds
Processing antigen E with L chain
Command executed successfully.
Interacting aminoacids computation took: 9.43 seconds


In [75]:
# Create output directory if it doesn’t exist
output_dir_path = os.path.join(out_dir, f"{config['Nano_source_pdb']}_{config['Antigen_complex_source_pdb']}")
os.makedirs(output_dir_path, exist_ok=True)

# Count existing JSON files in the directory that match the pattern
existing_files = [f for f in os.listdir(output_dir_path) if f.endswith('.json')]
ep = len(existing_files) + 1  # Set ep based on the count of existing JSON files

# Construct the JSON filename
json_file = os.path.join(output_dir_path, f"{config['Nano_source_pdb']}_{config['Antigen_complex_source_pdb']}_ep_{ep}.json")

# Copy nanobody_info to new_item
new_item = nanobody_info.copy()  # Create a shallow copy to modify

# Merge needed information into new_item
nano_scaffold_pdb = nanobody_info["pdb"]
antigen_source_pdb = temporal_info["pdb"]

new_item["pdb"] = f"{nano_scaffold_pdb}_{antigen_source_pdb}_ep_{ep}"
new_item["antigen_chains"] = temporal_info["antigen_chains"]
new_item["antigen_source"] = temporal_info["antigen_source"]

# Move "antigen_source" file to the newly generated output_dir_path and update the key/value
antigen_source_path = os.path.join(config["tmp_dir"], f"{temporal_info['antigen_source']}")
antigen_source_filename = os.path.basename(temporal_info['antigen_source'])
new_antigen_source_path = os.path.join(output_dir_path, antigen_source_filename)
shutil.move(antigen_source_path, new_antigen_source_path)
new_item["antigen_source"] = new_antigen_source_path

# Copy the "nano_source" file to the newly generated output_dir_path and update the key/value
nano_source_path = os.path.join(config["tmp_dir"], f"{config['Nano_source_pdb']}.pdb")
new_nano_source_path = os.path.join(output_dir_path, f"{config['Nano_source_pdb']}.pdb")
shutil.copy(nano_source_path, new_nano_source_path)
new_item["nano_source"] = new_nano_source_path

new_item["antigen_seqs"] = temporal_info["antigen_seqs"]
new_item["epitope_user_input"] = "yes"
new_item["epitope"] = epitope
new_item["numbering"] = config["numbering_scheme"]

# Delete everything in the config["tmp_dir"] directory
for filename in os.listdir(config["tmp_dir"]):
    file_path = os.path.join(config["tmp_dir"], filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # Remove the file or symbolic link
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # Remove the directory and all its contents
    except Exception as e:
        print(f"Failed to delete {file_path}. Reason: {e}")

# Save new_item to JSON
with open(json_file, 'w') as f:
    json.dump(new_item, f)

print(f"Saved nanobody information to: {json_file}")
print("Make sure the content of the newly generated json is not repeated")

Saved nanobody information to: /home/rioszemm/NanoDesigner/your_working_directory/denovo_epitope_info/7eow_8pwh/7eow_8pwh_ep_2.json
Make sure the content of the newly generated json is not repeated
